# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
**Task Type:** **Scoring / Ranking** (specifically risk/priority scoring to rank pages for content-refresh review).

Because content teams have limited review capacity, predicting a continuous **refresh-priority / decline-risk score** lets us rank pages from highest to lowest urgency rather than forcing a rigid yes/no cutoff too early. A score can still be thresholded later if needed, while keeping full ordering flexibility for a weekly refresh queue.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_framing = {
    "lane": "Content Refresh / Declining Performance Prioritization",
    "task_type": "scoring_and_ranking",
    "primary_unit": "page_or_page_period",
}
print("Framed task:", task_framing)


Framed task: {'lane': 'Content Refresh / Declining Performance Prioritization', 'task_type': 'scoring_and_ranking', 'primary_unit': 'page_or_page_period'}


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy:** A **proxy refresh-urgency / performance-decline signal** derived from observed search performance trends (such as drop/decline in impressions, clicks, or ranking momentum over subsequent windows), rather than a direct editorial label of whether a page was "actually refreshed."

Because historical datasets may not record every human editorial decision directly, using observed performance-decline signals as a proxy is practical—while remaining **directional / decision-support** until validated against real editorial outcomes and guarded against data leakage or lookahead bias.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Example sketch of how a binary/continuous proxy column could be defined once columns are confirmed
sample_proxy_df = pd.DataFrame({
    "impressions_prev": [1200, 450, 3000],
    "impressions_curr": [780, 460, 2950],
})
sample_proxy_df["impression_delta_pct"] = (
    (sample_proxy_df["impressions_curr"] - sample_proxy_df["impressions_prev"])
    / sample_proxy_df["impressions_prev"].replace(0, np.nan)
)
sample_proxy_df["proxy_decline_flag"] = (sample_proxy_df["impression_delta_pct"] <= -0.20).astype(int)
print(sample_proxy_df)


   impressions_prev  impressions_curr  impression_delta_pct  \
0              1200               780             -0.350000   
1               450               460              0.022222   
2              3000              2950             -0.016667   

   proxy_decline_flag  
0                   1  
1                   0  
2                   0  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric:** **Precision@K** (e.g., **Precision@20** or **Precision@50** depending on weekly editorial capacity), paired with **NDCG@K** or **Recall@K** as a secondary check.

Since editors can only review a fixed top-$K$ batch of pages per cycle, "good" means a high share of the top-$K$ ranked pages genuinely show real decline/refresh need (avoiding wasted audit hours), while not missing obvious high-impact decliners. Exact target thresholds will be calibrated after inspecting baseline data distributions and class/decline rates.

In [8]:
def precision_at_k(y_true, y_score, k=3):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(y_score))[:k]
    return float(np.mean(np.asarray(y_true)[order]))

demo_true = np.array([1, 0, 1, 1, 0])
demo_score = np.array([0.92, 0.81, 0.76, 0.40, 0.15])
print(f"Demo Precision@3: {precision_at_k(demo_true, demo_score, k=3):.3f}")

Demo Precision@3: 0.667


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** **One row = one page (or page-time slice)** in the lane's dataset slice, with a sketched target/priority column showing how a target value aligns with each row.

Below, the code loads the starter dataset from the repository/data source (with a clean fallback if a local CSV path differs slightly) and displays a real dataframe slice plus a sketched target column.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

candidate_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]

csv_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if csv_path is None:
    # If opened in Colab from a fresh runtime without repo files, fetch raw CSV from GitHub
    csv_path = "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df_lane = pd.read_csv(csv_path)
print("Loaded from:", csv_path)
print("Shape:", df_lane.shape)
print("Columns:", list(df_lane.columns))

# Sketch a target column on a slice of the real dataset
df_framed = df_lane.head(10).copy()
numeric_cols = df_framed.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    base_col = numeric_cols[0]
    median_val = df_framed[base_col].median()
    df_framed["sketched_refresh_priority_target"] = (df_framed[base_col] < median_val).astype(int)
else:
    df_framed["sketched_refresh_priority_target"] = 0

display(df_framed.head())

Loaded from: https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,sketched_refresh_priority_target
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats a Fixed Rule:** Search visibility and engagement patterns depend on multiple interacting signals (impressions, clicks, position shifts, query mix, and baseline scale) rather than a single static threshold. A hard-coded rule like `impressions < 500` or `clicks_drop > 20%` is brittle—it over-triggers on naturally low-volume or noisy pages and misses gradual multi-signal erosion on larger pages.

**Action Supported:** A ranked/scored list of pages helps content editors and SEO strategists decide **which pages to audit and refresh first** in their weekly workflow. All findings here are **observed / directional decision-support** inputs to guide human review rather than automated publishing changes.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick comparison check: fixed single-column threshold vs. multi-signal composite score
check_df = pd.DataFrame({
    "impressions": [300, 2500, 1200, 5000],
    "clicks": [40, 25, 18, 120],
    "position": [3.0, 14.5, 9.0, 2.2],
})
check_df["fixed_rule_low_imp"] = (check_df["impressions"] < 1000).astype(int)
# Higher composite score = stronger candidate for editorial review (illustrative directional score)
check_df["multi_signal_priority_score"] = (
    (check_df["position"] / check_df["position"].max()) * 0.5
    + (1 - (check_df["clicks"] / (check_df["clicks"].max() + 1))) * 0.5
).round(3)

print(check_df.sort_values("multi_signal_priority_score", ascending=False))


   impressions  clicks  position  fixed_rule_low_imp  \
1         2500      25      14.5                   0   
2         1200      18       9.0                   0   
0          300      40       3.0                   1   
3         5000     120       2.2                   0   

   multi_signal_priority_score  
1                        0.897  
2                        0.736  
0                        0.438  
3                        0.080  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.